# 3.3 位置编码 (Positional Encoding)

位置编码为Transformer注入位置信息，使其能够区分不同位置的token。

本节涵盖：
- 旋转位置编码（RoPE）
- ALiBi
- 正弦位置编码

## 1. 旋转位置编码（RoPE）

**目的**：为注意力计算注入相对位置信息

**基本原理**：将位置信息编码为旋转矩阵作用于Q和K，使内积结果自然包含相对位置信息，具有良好的外推性。代表：LLaMA、Qwen、Mistral。

**核心公式**：RoPE将位置m的向量[x₁, x₂]旋转角度mθ：
- [x₁, x₂] → [x₁cos(mθ) - x₂sin(mθ), x₁sin(mθ) + x₂cos(mθ)]
- Q·K^T 自然包含相对位置 (m-n) 的信息

In [ ]:
import torch
import torch.nn as nn
import math

torch.manual_seed(42)

class RotaryPositionEmbedding(nn.Module):
    def __init__(self, d_head=16, max_seq_len=512, base=10000):
        super().__init__()
        inv_freq = 1.0 / (base ** (torch.arange(0, d_head, 2).float() / d_head))
        self.register_buffer('inv_freq', inv_freq)
        self.d_head = d_head

    def forward(self, x, seq_len=None):
        if seq_len is None:
            seq_len = x.size(1)
        t = torch.arange(seq_len, device=x.device, dtype=self.inv_freq.dtype)
        freqs = torch.outer(t, self.inv_freq)
        emb = torch.cat([freqs, freqs], dim=-1)
        cos = emb.cos().unsqueeze(0).unsqueeze(0)
        sin = emb.sin().unsqueeze(0).unsqueeze(0)
        return cos, sin

def apply_rotary_emb(x, cos, sin):
    x1, x2 = x[..., :x.shape[-1]//2], x[..., x.shape[-1]//2:]
    rotated = torch.cat([-x2, x1], dim=-1)
    return x * cos + rotated * sin

d_head = 16
rope = RotaryPositionEmbedding(d_head=d_head)
x = torch.randn(2, 10, 4, d_head)
cos, sin = rope(x, seq_len=10)
x_rotated = apply_rotary_emb(x, cos, sin)

print('=== Rotary Position Embedding (RoPE) ===')
print(f'Input shape: {x.shape}')
print(f'cos shape: {cos.shape}, sin shape: {sin.shape}')
print(f'Rotated output shape: {x_rotated.shape}')

q = x_rotated[0, :, 0, :]
for dist in [0, 1, 3, 5, 9]:
    i, j = 0, dist
    dot = (q[i] * q[j]).sum().item()
    norm = q[i].norm().item() * q[j].norm().item()
    cos_sim = dot / (norm + 1e-8)
    print(f'  pos 0 vs pos {dist}: dot={dot:.3f}, cos_sim={cos_sim:.3f}')

print('\nKey: RoPE encodes relative position through rotation,')
print('making attention naturally distance-aware without explicit position tokens.')

## 2. ALiBi（Attention with Linear Biases）

**目的**：实现长度外推

**基本原理**：不在嵌入层添加位置编码，而是在注意力分数上添加与距离成比例的线性偏置，使模型能处理比训练时更长的序列。

**ALiBi偏置**：bias[i][j] = -m * |i - j|，其中m是每个头的斜率
- 距离越远，偏置越负，注意力权重越低
- 不同头使用不同的斜率m，捕获不同尺度的位置关系
- 推理时可直接应用于更长序列，无需修改

In [ ]:
import torch
import torch.nn as nn
import math

torch.manual_seed(42)

class ALiBiPositionBias(nn.Module):
    def __init__(self, n_heads=8, max_seq_len=512):
        super().__init__()
        slopes = torch.tensor([2 ** (-8 * i / n_heads) for i in range(n_heads)])
        self.register_buffer('slopes', slopes)
        positions = torch.arange(max_seq_len)
        dists = positions.unsqueeze(0) - positions.unsqueeze(1)
        dists = dists.abs().float()
        bias = -self.slopes.unsqueeze(1).unsqueeze(2) * dists.unsqueeze(0)
        self.register_buffer('bias', bias)

    def forward(self, seq_len):
        return self.bias[:, :seq_len, :seq_len]

n_heads = 8
alibi = ALiBiPositionBias(n_heads=n_heads, max_seq_len=16)
bias = alibi(8)

print('=== ALiBi (Attention with Linear Biases) ===')
print(f'Heads: {n_heads}, Sequence length: 8')
print(f'\nSlopes per head: {[f"{s:.4f}" for s in alibi.slopes.tolist()]}')
print(f'\nBias for head 0 (steepest slope):')
for i in range(4):
    row = [f'{bias[0, i, j]:6.3f}' for j in range(4)]
    print(f'  pos {i}: [{"  ".join(row)}]')

print(f'\nBias for head 7 (gentlest slope):')
for i in range(4):
    row = [f'{bias[7, i, j]:6.3f}' for j in range(4)]
    print(f'  pos {i}: [{"  ".join(row)}]')

print('\nKey: ALiBi adds distance-proportional negative bias to attention scores.')
print('Farther positions get more negative bias, reducing attention to distant tokens.')
print('This enables length extrapolation without any position embedding modification.')

## 3. 正弦位置编码

**目的**：原始Transformer的位置编码方式

**基本原理**：使用不同频率的正弦和余弦函数生成位置编码，通过加法注入到输入嵌入中，不同频率捕获不同尺度的位置关系。

**公式**：
- PE(pos, 2i) = sin(pos / 10000^(2i/d))
- PE(pos, 2i+1) = cos(pos / 10000^(2i/d))

**特点**：
- 相对位置可通过线性变换从绝对位置编码推导
- 不同频率对应不同尺度的位置关系
- 是最经典的位置编码方式，但外推性不如RoPE和ALiBi

In [ ]:
import torch
import math

def sinusoidal_position_encoding(max_len=512, d_model=64):
    pe = torch.zeros(max_len, d_model)
    position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
    div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
    pe[:, 0::2] = torch.sin(position * div_term)
    pe[:, 1::2] = torch.cos(position * div_term)
    return pe

d_model = 64
max_len = 128
pe = sinusoidal_position_encoding(max_len, d_model)

print('=== Sinusoidal Position Encoding ===')
print(f'Shape: {pe.shape} (max_len={max_len}, d_model={d_model})')
print(f'\nPosition 0: first 8 dims = {[f"{v:.3f}" for v in pe[0, :8].tolist()]}')
print(f'Position 1: first 8 dims = {[f"{v:.3f}" for v in pe[1, :8].tolist()]}')
print(f'Position 50: first 8 dims = {[f"{v:.3f}" for v in pe[50, :8].tolist()]}')

similarities = []
for dist in [1, 5, 10, 50, 100]:
    sim = torch.nn.functional.cosine_similarity(pe[0].unsqueeze(0), pe[dist].unsqueeze(0)).item()
    similarities.append((dist, sim))
print(f'\nCosine similarity from position 0:')
for dist, sim in similarities:
    print(f'  pos 0 vs pos {dist:3d}: {sim:.4f}')

print(f'\nKey: Low-frequency dimensions change slowly (capture long-range position),')
print(f'high-frequency dimensions change quickly (capture short-range position).')
print(f'Similarity generally decreases with distance, but not monotonically due to periodicity.')